# Notebook 09 — Final Pipeline Export
## Section 1: Save Final Model on Disk

Load the final model config from Notebook 08, retrain on **train data (Jul+Aug)**, and save:

- `final_rf_model.joblib` — loadable model pipeline  
- `final_model_export_report.json` — export metadata  

**Target:** `delay_in_min` | **Metric:** MAE

In [1]:
# =============================================================================
# Notebook 09 | Section 1: Save Final Model on Disk
# =============================================================================
# Load final_model_selection.json → retrain tuned RF on train → save joblib.
# =============================================================================

from __future__ import annotations

import json
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore", category=FutureWarning)

# Speed vs full train — set False for full 255k rows (slower)
USE_TRAIN_SAMPLE = True
TRAIN_SAMPLE_SIZE = 120_000
RANDOM_STATE = 42

PRIMARY_TARGET = "delay_in_min"
CATEGORICAL_COLS = ["eva", "train_number", "final_destination_station"]


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        config_file = candidate / "data" / "reference" / "project_config.json"
        if config_file.exists():
            return candidate
        nb = candidate / "Notebooks"
        if (nb / "data" / "reference" / "project_config.json").exists():
            return nb
    raise FileNotFoundError("Cannot find data/reference/project_config.json")


PROJECT_ROOT = find_project_root()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROCESSED_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

FINAL_SEL_PATH = REFERENCE_DIR / "final_model_selection.json"
TRAIN_PATH = PROCESSED_DIR / "ice_train.parquet"
TEST_PATH = PROCESSED_DIR / "ice_test.parquet"

MODEL_PATH = MODELS_DIR / "final_rf_model.joblib"
EXPORT_REPORT_PATH = REFERENCE_DIR / "final_model_export_report.json"


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def make_json_safe(obj):
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj


def build_preprocessor(feature_cols: list[str]) -> ColumnTransformer:
    cat_cols = [c for c in CATEGORICAL_COLS if c in feature_cols]
    num_cols = [c for c in feature_cols if c not in cat_cols]
    return ColumnTransformer(
        transformers=[
            ("num", SimpleImputer(strategy="median"), num_cols),
            (
                "cat",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
                ]),
                cat_cols,
            ),
        ]
    )


# =============================================================================
# LOAD FINAL CONFIG + DATA
# =============================================================================
final_sel = load_json(FINAL_SEL_PATH)
fm = final_sel["final_model"]

feature_cols = fm["feature_columns"]
best_params = fm["best_params"]

print("Notebook 09 | Section 1 — Save final model")
print(f"Algorithm   : {fm['algorithm']}")
print(f"Feature set : {fm['feature_set']}")
print(f"Features    : {len(feature_cols)}")
print(f"Best params : {best_params}")
print(f"Expected MAE: {fm['test_mae']:.4f} min (from NB08 Sep test)")
print()

train_df = pd.read_parquet(TRAIN_PATH)
test_df = pd.read_parquet(TEST_PATH)

if USE_TRAIN_SAMPLE and len(train_df) > TRAIN_SAMPLE_SIZE:
    fit_df = train_df.sample(n=TRAIN_SAMPLE_SIZE, random_state=RANDOM_STATE)
    print(f"Training sample: {len(fit_df):,} / {len(train_df):,} train rows")
else:
    fit_df = train_df
    print(f"Training on full train: {len(fit_df):,} rows")

print(f"Test rows (verify): {len(test_df):,}")
print()

# =============================================================================
# BUILD + TRAIN FINAL PIPELINE
# =============================================================================
rf_params = {**best_params, "random_state": RANDOM_STATE, "n_jobs": -1}
final_pipeline = Pipeline([
    ("prep", build_preprocessor(feature_cols)),
    ("model", RandomForestRegressor(**rf_params)),
])

X_train = fit_df[feature_cols]
y_train = fit_df[PRIMARY_TARGET].values
X_test = test_df[feature_cols]
y_test = test_df[PRIMARY_TARGET].values

print("Training final model...")
final_pipeline.fit(X_train, y_train)

train_mae = round(float(mean_absolute_error(y_train, final_pipeline.predict(X_train))), 4)
test_mae = round(float(mean_absolute_error(y_test, final_pipeline.predict(X_test))), 4)

print(f"  Train MAE: {train_mae:.4f} min")
print(f"  Test MAE : {test_mae:.4f} min  (should be close to NB08: {fm['test_mae']:.4f})")
print()

# =============================================================================
# SAVE MODEL + REPORT
# =============================================================================
joblib.dump(final_pipeline, MODEL_PATH)

export_report = {
    "metadata": {
        "notebook": "09_Final_Pipeline_Export",
        "section": "Section 1",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "primary_target": PRIMARY_TARGET,
        "primary_metric": "mae",
        "task_type": "regression",
    },
    "final_model": {
        "algorithm": fm["algorithm"],
        "feature_set": fm["feature_set"],
        "feature_columns": feature_cols,
        "best_params": best_params,
        "source_selection_file": str(FINAL_SEL_PATH),
    },
    "training": {
        "train_parquet": str(TRAIN_PATH),
        "train_months": ["2024-07", "2024-08"],
        "rows_used_for_fit": int(len(fit_df)),
        "use_train_sample": USE_TRAIN_SAMPLE,
        "train_sample_size": TRAIN_SAMPLE_SIZE if USE_TRAIN_SAMPLE else None,
    },
    "evaluation_on_reload": {
        "test_parquet": str(TEST_PATH),
        "test_month": "2024-09",
        "train_mae": train_mae,
        "test_mae": test_mae,
        "nb08_reference_test_mae": fm["test_mae"],
    },
    "saved_files": {
        "model_joblib": str(MODEL_PATH),
        "export_report": str(EXPORT_REPORT_PATH),
    },
    "how_to_load": (
        "import joblib; pipe = joblib.load('"
        + str(MODEL_PATH).replace("\\", "/")
        + "'); pipe.predict(X[feature_columns])"
    ),
}

with EXPORT_REPORT_PATH.open("w", encoding="utf-8") as f:
    json.dump(make_json_safe(export_report), f, indent=2, ensure_ascii=False)

# Quick demo: 3 test predictions
demo = test_df.head(3)[feature_cols + [PRIMARY_TARGET]].copy()
demo["predicted_delay_min"] = final_pipeline.predict(demo[feature_cols]).round(2)

print("=" * 72)
print("Section 1 COMPLETE")
print("=" * 72)
print(f"Model saved : {MODEL_PATH}")
print(f"Report saved: {EXPORT_REPORT_PATH}")
print()
print("Demo predictions (first 3 test rows):")
print(demo[[PRIMARY_TARGET, "predicted_delay_min"]].to_string(index=False))
print()
print("Next: Section 2 — full pipeline summary + reproduce guide")
print("=" * 72)

Notebook 09 | Section 1 — Save final model
Algorithm   : RandomForestRegressor
Feature set : full_with_weather
Features    : 15
Best params : {'n_estimators': 100, 'max_depth': 20, 'min_samples_leaf': 10, 'max_samples': 0.5}
Expected MAE: 10.4196 min (from NB08 Sep test)

Training sample: 120,000 / 255,062 train rows
Test rows (verify): 121,964

Training final model...
  Train MAE: 9.4993 min
  Test MAE : 10.4196 min  (should be close to NB08: 10.4196)

Section 1 COMPLETE
Model saved : c:\Users\Manikanta\Desktop\ML Project\Notebooks\data\processed\models\final_rf_model.joblib
Report saved: c:\Users\Manikanta\Desktop\ML Project\Notebooks\data\reference\final_model_export_report.json

Demo predictions (first 3 test rows):
 delay_in_min  predicted_delay_min
            0                11.36
           -7                14.88
            0                20.30

Next: Section 2 — full pipeline summary + reproduce guide


# Notebook 09 — Final Pipeline Export
## Section 2: Full Pipeline Summary & Reproduce Guide

One master document: all notebooks, all saved Parquet files, final results, and how to reproduce.

**Output:** `full_project_pipeline_summary.json`

In [2]:
# =============================================================================
# Notebook 09 | Section 2: Full Pipeline Summary & Reproduce Guide
# =============================================================================
# Compile end-to-end pipeline doc from saved JSON + parquet paths on disk.
# =============================================================================

from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        config_file = candidate / "data" / "reference" / "project_config.json"
        if config_file.exists():
            return candidate
        nb = candidate / "Notebooks"
        if (nb / "data" / "reference" / "project_config.json").exists():
            return nb
    raise FileNotFoundError("Cannot find data/reference/project_config.json")


PROJECT_ROOT = find_project_root()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROCESSED_DIR / "models"
FIGURES_DIR = PROCESSED_DIR / "figures"

SUMMARY_PATH = REFERENCE_DIR / "full_project_pipeline_summary.json"


def load_json(path: Path) -> dict:
    if not path.exists():
        return None
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def make_json_safe(obj):
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj


def parquet_info(path: Path) -> dict:
    if not path.exists():
        return {"exists": False, "path": str(path)}
    rows = int(len(pd.read_parquet(path, columns=["id"] if "id" in pd.read_parquet(path, columns=[]).columns else [])))
    # fallback row count
    try:
        rows = int(len(pd.read_parquet(path, columns=["id"])))
    except Exception:
        rows = int(len(pd.read_parquet(path)))
    return {
        "exists": True,
        "path": str(path),
        "size_mb": round(path.stat().st_size / 1e6, 2),
        "rows": rows,
    }


config = load_json(REFERENCE_DIR / "project_config.json")
final_sel = load_json(REFERENCE_DIR / "final_model_selection.json")
export_report = load_json(REFERENCE_DIR / "final_model_export_report.json")
ablation = load_json(REFERENCE_DIR / "weather_ablation_report.json")

TARGET_MONTHS = config["scope"]["target_months"]

print("Notebook 09 | Section 2 — Full pipeline summary")
print()

# =============================================================================
# PARQUET INVENTORY
# =============================================================================
parquet_files = {
    "raw_ice": [f"ice_raw_{m}.parquet" for m in TARGET_MONTHS],
    "standardized": [f"ice_standardized_{m}.parquet" for m in TARGET_MONTHS],
    "cleaned": [f"ice_cleaned_{m}.parquet" for m in TARGET_MONTHS],
    "weather": [f"weather_by_station_{m}.parquet" for m in TARGET_MONTHS],
    "merged": [f"ice_weather_merged_{m}.parquet" for m in TARGET_MONTHS],
    "modeling_ready": [f"ice_modeling_ready_{m}.parquet" for m in TARGET_MONTHS],
    "train_test": ["ice_train.parquet", "ice_test.parquet"],
    "combined": ["ice_modeling_ready_all_months.parquet"],
}

inventory = {}
for category, files in parquet_files.items():
    inventory[category] = {}
    for fname in files:
        inventory[category][fname] = parquet_info(PROCESSED_DIR / fname)

print("=" * 72)
print("PARQUET FILES ON DISK")
print("=" * 72)
for category, files in inventory.items():
    print(f"\n{category}:")
    for fname, info in files.items():
        if info["exists"]:
            print(f"  [OK] {fname}  ({info['rows']:,} rows, {info['size_mb']} MB)")
        else:
            print(f"  [--] {fname}  (missing)")

# =============================================================================
# PIPELINE STEPS
# =============================================================================
pipeline_steps = [
    {
        "notebook": "01",
        "title": "Project Definition & Data Understanding",
        "actions": [
            "Define scope: ICE only, regression on delay_in_min, MAE, Jul-Sep 2024",
            "Save project_config.json, research_framework.json, merge_strategy.json",
        ],
        "outputs": ["data/reference/*.json"],
    },
    {
        "notebook": "02",
        "title": "Data Acquisition",
        "actions": [
            "Download ICE data from Hugging Face (3 months)",
            "Profile delays, cancellations, weather demo",
        ],
        "outputs": ["ice_raw_YYYY-MM.parquet"],
    },
    {
        "notebook": "03",
        "title": "Cleaning & Standardization",
        "actions": [
            "Timestamps → Europe/Berlin + hour merge key",
            "Drop canceled stops, fill station names",
        ],
        "outputs": ["ice_standardized_*.parquet", "ice_cleaned_*.parquet"],
    },
    {
        "notebook": "04",
        "title": "Data Integration",
        "actions": [
            "Geocode eva → lat/lon (Nominatim)",
            "Fetch Open-Meteo hourly weather",
            "Left join on eva + departure hour",
        ],
        "outputs": [
            "station_coordinates.json",
            "weather_by_station_*.parquet",
            "ice_weather_merged_*.parquet",
        ],
    },
    {
        "notebook": "05",
        "title": "Exploratory Data Analysis",
        "actions": [
            "Delay distribution (median ~4, mean ~11 min)",
            "Weather vs delay correlations (weak in summer)",
        ],
        "outputs": ["eda_*_report_multi_month.json", "figures/*.png"],
    },
    {
        "notebook": "06",
        "title": "Feature Engineering",
        "actions": [
            "Build time + operational + weather features",
            "Drop visibility (100% null), time split train/test",
        ],
        "outputs": [
            "ice_modeling_ready_*.parquet",
            "ice_train.parquet (Jul+Aug)",
            "ice_test.parquet (Sep)",
            "feature_manifest.json",
        ],
    },
    {
        "notebook": "07",
        "title": "Baseline Models",
        "actions": [
            "Naive mean/median + Linear/Ridge + Random Forest",
            "Compare operational vs +weather feature sets",
        ],
        "outputs": ["baseline_results.json", "random_forest_results.json"],
    },
    {
        "notebook": "08",
        "title": "Tuning & Evaluation",
        "actions": [
            "RF hyperparameter tuning (CV on train)",
            "Formal weather ablation + final model selection",
        ],
        "outputs": [
            "rf_tuning_results.json",
            "weather_ablation_report.json",
            "final_model_selection.json",
        ],
    },
    {
        "notebook": "09",
        "title": "Final Pipeline Export",
        "actions": [
            "Save final_rf_model.joblib",
            "Full pipeline summary for report/viva",
        ],
        "outputs": [
            "final_rf_model.joblib",
            "full_project_pipeline_summary.json",
        ],
    },
]

# =============================================================================
# KEY RESULTS
# =============================================================================
merged_total = sum(
    inventory["merged"][f"ice_weather_merged_{m}.parquet"]["rows"]
    for m in TARGET_MONTHS
    if inventory["merged"][f"ice_weather_merged_{m}.parquet"]["exists"]
)

key_results = {
    "primary_task": "regression on delay_in_min",
    "primary_metric": "mae",
    "target_months": TARGET_MONTHS,
    "train_months": ["2024-07", "2024-08"],
    "test_month": "2024-09",
    "merged_rows_total": merged_total,
    "train_rows": inventory["train_test"]["ice_train.parquet"].get("rows"),
    "test_rows": inventory["train_test"]["ice_test.parquet"].get("rows"),
    "final_model": final_sel["final_model"] if final_sel else None,
    "naive_median_test_mae": final_sel["benchmarks"]["naive_median_test_mae"] if final_sel else None,
    "weather_ablation_gain_min": ablation["ablation"]["weather_gain_min"] if ablation else None,
    "rq_answers": final_sel["rq_answers"] if final_sel else None,
}

print()
print("=" * 72)
print("KEY RESULTS")
print("=" * 72)
print(f"  Task            : regression on delay_in_min")
print(f"  Metric          : MAE")
print(f"  Merged rows     : {merged_total:,}")
print(f"  Train / Test    : {key_results['train_rows']:,} / {key_results['test_rows']:,}")
if final_sel:
    print(f"  Final ML MAE    : {final_sel['final_model']['test_mae']:.4f} min (Sep)")
    print(f"  Naive median    : {final_sel['benchmarks']['naive_median_test_mae']:.4f} min")
    print(f"  Weather gain    : {ablation['ablation']['weather_gain_min']:.4f} min")
print()

# =============================================================================
# REPRODUCE GUIDE
# =============================================================================
reproduce_guide = {
    "step_1": "Run Notebooks 01-09 in order from Notebooks/ folder",
    "step_2": "All data reloads from data/processed/*.parquet (not from memory)",
    "step_3": "Merged data: ice_weather_merged_YYYY-MM.parquet",
    "step_4": "Modeling: ice_train.parquet → train, ice_test.parquet → test",
    "step_5": "Load saved model: joblib.load('data/processed/models/final_rf_model.joblib')",
    "merge_key": "eva + departure_planned_hour_naive (planned departure floored to hour)",
    "professor_requirements": {
        "regression_only": True,
        "mae_only": True,
        "merged_data_on_disk": True,
        "weather_ablation_done": True,
        "time_based_split": "train Jul+Aug, test Sep",
    },
}

# =============================================================================
# SAVE MASTER SUMMARY
# =============================================================================
summary = {
    "metadata": {
        "project_title": config["project"]["title"],
        "authors": config["project"]["authors"],
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "notebook": "09_Final_Pipeline_Export",
        "section": "Section 2",
        "status": "PROJECT COMPLETE",
    },
    "professor_aligned": {
        "regression_only": True,
        "target": "delay_in_min",
        "primary_metric": "mae",
        "no_classification_models": True,
        "no_rmse_required": True,
        "data_saved_on_disk_parquet": True,
    },
    "data_storage": {
        "root": str(PROCESSED_DIR),
        "reference_configs": str(REFERENCE_DIR),
        "models": str(MODELS_DIR),
        "figures": str(FIGURES_DIR),
        "parquet_inventory": inventory,
        "why_parquet": "Compressed, fast, preserves types, reloadable across notebooks",
    },
    "pipeline_steps": pipeline_steps,
    "key_results": key_results,
    "research_questions": final_sel["rq_answers"] if final_sel else {},
    "honest_conclusion": final_sel.get("honest_conclusion") if final_sel else "",
    "reproduce_guide": reproduce_guide,
    "important_files_for_viva": {
        "merged_data": [f"data/processed/ice_weather_merged_{m}.parquet" for m in TARGET_MONTHS],
        "train_test": ["data/processed/ice_train.parquet", "data/processed/ice_test.parquet"],
        "final_model": "data/processed/models/final_rf_model.joblib",
        "final_results": "data/reference/final_model_selection.json",
        "weather_ablation": "data/reference/weather_ablation_report.json",
        "this_summary": str(SUMMARY_PATH),
    },
}

with SUMMARY_PATH.open("w", encoding="utf-8") as f:
    json.dump(make_json_safe(summary), f, indent=2, ensure_ascii=False)

print("=" * 72)
print("Section 2 COMPLETE")
print("=" * 72)
print(f"Saved master summary: {SUMMARY_PATH}")
print()
print("This file contains:")
print("  - All Parquet paths + row counts")
print("  - Full notebook pipeline (01-09)")
print("  - Final MAE results + RQ answers")
print("  - Reproduce guide for viva/report")
print()
print("Next: Section 3 — final close-out")
print("=" * 72)

Notebook 09 | Section 2 — Full pipeline summary

PARQUET FILES ON DISK

raw_ice:
  [OK] ice_raw_2024-07.parquet  (157,886 rows, 5.5 MB)
  [OK] ice_raw_2024-08.parquet  (150,289 rows, 5.3 MB)
  [OK] ice_raw_2024-09.parquet  (149,218 rows, 5.29 MB)

standardized:
  [OK] ice_standardized_2024-07.parquet  (157,886 rows, 5.67 MB)
  [OK] ice_standardized_2024-08.parquet  (150,289 rows, 5.46 MB)
  [OK] ice_standardized_2024-09.parquet  (149,218 rows, 5.44 MB)

cleaned:
  [OK] ice_cleaned_2024-07.parquet  (146,389 rows, 5.46 MB)
  [OK] ice_cleaned_2024-08.parquet  (136,321 rows, 5.1 MB)
  [OK] ice_cleaned_2024-09.parquet  (135,547 rows, 5.06 MB)

weather:
  [OK] weather_by_station_2024-07.parquet  (70,680 rows, 0.31 MB)
  [OK] weather_by_station_2024-08.parquet  (72,912 rows, 0.31 MB)
  [OK] weather_by_station_2024-09.parquet  (70,560 rows, 0.32 MB)

merged:
  [OK] ice_weather_merged_2024-07.parquet  (146,389 rows, 6.25 MB)
  [OK] ice_weather_merged_2024-08.parquet  (136,321 rows, 5.82 MB)
  [

# Notebook 09 — Final Pipeline Export
## Section 3: Project Close-Out

Final checklist: model on disk, pipeline summary saved, project complete.

**Status:** ICE Delay Prediction capstone — DONE

In [3]:
# =============================================================================
# Notebook 09 | Section 3: Final Project Close-Out
# =============================================================================
# Verify all deliverables; save notebook_09_summary.json + project COMPLETE flag
# =============================================================================

from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        config_file = candidate / "data" / "reference" / "project_config.json"
        if config_file.exists():
            return candidate
        nb = candidate / "Notebooks"
        if (nb / "data" / "reference" / "project_config.json").exists():
            return nb
    raise FileNotFoundError("Cannot find data/reference/project_config.json")


PROJECT_ROOT = find_project_root()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROCESSED_DIR / "models"
FIGURES_DIR = PROCESSED_DIR / "figures"

SUMMARY_PATH = REFERENCE_DIR / "notebook_09_summary.json"
PIPELINE_SUMMARY = REFERENCE_DIR / "full_project_pipeline_summary.json"
PROJECT_COMPLETE = REFERENCE_DIR / "project_complete.json"


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def make_json_safe(obj):
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj


config = load_json(REFERENCE_DIR / "project_config.json")
pipeline = load_json(PIPELINE_SUMMARY)
final_sel = load_json(REFERENCE_DIR / "final_model_selection.json")

print("=" * 72)
print("ICE TRAIN DELAY PREDICTION — FINAL CLOSE-OUT")
print("=" * 72)
print(f"Project : {config['project']['title']}")
print(f"Authors : {', '.join(config['project']['authors'])}")
print(f"Months  : {', '.join(config['scope']['target_months'])}")
print(f"Target  : {config['ml_tasks']['primary']['target']}")
print(f"Metric  : {config['ml_tasks']['primary']['primary_metric'].upper()}")
print()

checklist = []


def check(label: str, path: Path) -> bool:
    ok = path.exists()
    checklist.append({"label": label, "path": str(path), "exists": ok})
    print(f"  [{'OK' if ok else 'MISSING'}] {label}")
    return ok


print("DELIVERABLES CHECKLIST")
print("-" * 72)

all_ok = True

print("\n[Data on disk — Parquet]")
for m in config["scope"]["target_months"]:
    all_ok &= check(f"ice_weather_merged_{m}.parquet", PROCESSED_DIR / f"ice_weather_merged_{m}.parquet")
all_ok &= check("ice_train.parquet", PROCESSED_DIR / "ice_train.parquet")
all_ok &= check("ice_test.parquet", PROCESSED_DIR / "ice_test.parquet")

print("\n[Model & results]")
all_ok &= check("final_rf_model.joblib", MODELS_DIR / "final_rf_model.joblib")
all_ok &= check("final_model_selection.json", REFERENCE_DIR / "final_model_selection.json")
all_ok &= check("weather_ablation_report.json", REFERENCE_DIR / "weather_ablation_report.json")
all_ok &= check("full_project_pipeline_summary.json", PIPELINE_SUMMARY)
all_ok &= check("final_model_mae_comparison.png", FIGURES_DIR / "final_model_mae_comparison.png")

print("\n[Notebook summaries]")
for nb in ["01", "02", "03", "04", "05", "06", "07", "08"]:
    fname = f"notebook_{nb}_summary.json"
    path = REFERENCE_DIR / fname
    if path.exists():
        check(fname, path)
    # not all may exist — don't fail on missing early summaries

print()
print("=" * 72)
print("FINAL RESULTS")
print("=" * 72)
fm = final_sel["final_model"]
bench = final_sel["benchmarks"]
print(f"  Final model        : {fm['algorithm']} ({fm['feature_set']})")
print(f"  Final test MAE     : {fm['test_mae']:.4f} min  (September 2024)")
print(f"  Naive median MAE   : {bench['naive_median_test_mae']:.4f} min")
print(f"  Weather ablation   : {pipeline['key_results']['weather_ablation_gain_min']:.4f} min gain")
print()
print("  RQ1 (operational)  : Partial — RF beats linear, not naive median")
print("  RQ2 (weather)      : No meaningful MAE gain in Jul-Sep")
print("  RQ3 (which weather): Temperature & wind — weak overall effect")
print()
print("  Professor requirements met:")
print("    [x] Regression only (delay_in_min)")
print("    [x] MAE only")
print("    [x] Merged data saved as Parquet on disk")
print("    [x] With vs without weather ablation done")
print("    [x] Time-based train/test split (Jul+Aug / Sep)")
print()

if not all_ok:
    missing = [c["label"] for c in checklist if not c["exists"]]
    print(f"WARNING — missing: {missing}")
else:
    print("ALL CORE DELIVERABLES: OK")

# =============================================================================
# SAVE CLOSE-OUT FILES
# =============================================================================
nb09_summary = {
    "metadata": {
        "notebook": "09_Final_Pipeline_Export",
        "section": "Section 3",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "status": "PROJECT COMPLETE",
    },
    "checklist": checklist,
    "all_core_deliverables_ok": all_ok,
    "final_model_test_mae": fm["test_mae"],
    "naive_median_test_mae": bench["naive_median_test_mae"],
    "pipeline_summary_file": str(PIPELINE_SUMMARY),
    "model_file": str(MODELS_DIR / "final_rf_model.joblib"),
}

with SUMMARY_PATH.open("w", encoding="utf-8") as f:
    json.dump(make_json_safe(nb09_summary), f, indent=2, ensure_ascii=False)

project_complete = {
    "status": "COMPLETE",
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "project_title": config["project"]["title"],
    "authors": config["project"]["authors"],
    "notebooks": "01 through 09",
    "primary_task": "regression on delay_in_min",
    "primary_metric": "mae",
    "months": config["scope"]["target_months"],
    "final_ml_test_mae_min": fm["test_mae"],
    "best_overall_test_mae_min": bench["naive_median_test_mae"],
    "best_overall_method": "NaiveMedian",
    "final_ml_model": "Tuned RandomForestRegressor (full_with_weather)",
    "data_root": str(PROCESSED_DIR),
    "master_summary": str(PIPELINE_SUMMARY),
    "one_line_conclusion": final_sel["honest_conclusion"],
}

with PROJECT_COMPLETE.open("w", encoding="utf-8") as f:
    json.dump(make_json_safe(project_complete), f, indent=2, ensure_ascii=False)

print()
print("=" * 72)
print("PROJECT COMPLETE")
print("=" * 72)
print(f"Saved: {SUMMARY_PATH}")
print(f"Saved: {PROJECT_COMPLETE}")
print()
print("You finished Notebooks 01-09.")
print("Use full_project_pipeline_summary.json for report/viva.")
print("=" * 72)

ICE TRAIN DELAY PREDICTION — FINAL CLOSE-OUT
Project : ICE Train Delay Prediction Using Railway Operational Data and Historical Weather Data
Authors : Manikanta Engalligi, abhinay-sambherao
Months  : 2024-07, 2024-08, 2024-09
Target  : delay_in_min
Metric  : MAE

DELIVERABLES CHECKLIST
------------------------------------------------------------------------

[Data on disk — Parquet]
  [OK] ice_weather_merged_2024-07.parquet
  [OK] ice_weather_merged_2024-08.parquet
  [OK] ice_weather_merged_2024-09.parquet
  [OK] ice_train.parquet
  [OK] ice_test.parquet

[Model & results]
  [OK] final_rf_model.joblib
  [OK] final_model_selection.json
  [OK] weather_ablation_report.json
  [OK] full_project_pipeline_summary.json
  [OK] final_model_mae_comparison.png

[Notebook summaries]
  [OK] notebook_02_summary.json
  [OK] notebook_03_summary.json
  [OK] notebook_04_summary.json
  [OK] notebook_05_summary.json
  [OK] notebook_06_summary.json
  [OK] notebook_07_summary.json
  [OK] notebook_08_summary.